# 🧠 Alzheimer's Adaptive Activity Recommendation Engine
**ML Backend Pipeline** · Dataset · Scoring · Feature Engineering · Recommendation Model · API Output

> This notebook implements the full backend logic for a cognitive activity recommendation system
> designed for Alzheimer's patients. It is structured in 8 sections, from data loading to
> model training and API-ready output generation.

## 1. Imports & Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────
import json
import warnings
import random
from datetime import datetime, timedelta
from collections import defaultdict
from statistics import mean, stdev

warnings.filterwarnings('ignore')

# ── Data & ML ─────────────────────────────────────────────────
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed — XGB section will be skipped.")
    print("Install with: pip install xgboost")

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})

print("✅ All imports successful.")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")


## 2. Activity Dataset
Load the 25-activity CSV and explore the schema. Each row defines one cognitive exercise
with its difficulty tier, scoring weights, and target cognitive skill.

In [ ]:
# ── Load dataset ──────────────────────────────────────────────
df = pd.read_csv('alzheimer_activities.csv')

print(f"Shape : {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


In [ ]:
# ── Basic statistics ──────────────────────────────────────────
print("\n=== Difficulty distribution ===")
print(df['difficulty_level'].value_counts())

print("\n=== Category distribution ===")
print(df['category'].value_counts())

print("\n=== Score range by difficulty ===")
print(df.groupby('difficulty_level')['base_score'].describe().round(2))


In [ ]:
# ── Visualise dataset ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Base score by difficulty
order = ['Low', 'Medium', 'High']
colors = ['#4CAF50', '#FF9800', '#F44336']
df_plot = df.copy()
df_plot['difficulty_level'] = pd.Categorical(df_plot['difficulty_level'], categories=order, ordered=True)
df_sorted = df_plot.sort_values('difficulty_level')

axes[0].bar(
    [f"{r['activity_name'][:12]}" for _, r in df_sorted.iterrows()],
    df_sorted['base_score'],
    color=[colors[order.index(d)] for d in df_sorted['difficulty_level']]
)
axes[0].set_title('Base Score per Activity')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90, fontsize=7)
axes[0].set_ylabel('Base Score')

# Activities per category
cat_counts = df['category'].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color='steelblue')
axes[1].set_title('Activities per Category')
axes[1].set_xlabel('Count')

# Expected time by difficulty
for i, (lvl, col) in enumerate(zip(order, colors)):
    sub = df[df['difficulty_level'] == lvl]['expected_time_sec']
    axes[2].scatter([i]*len(sub), sub, color=col, s=80, label=lvl, alpha=0.8, zorder=3)
axes[2].set_xticks([0, 1, 2])
axes[2].set_xticklabels(order)
axes[2].set_title('Expected Time by Difficulty')
axes[2].set_ylabel('Seconds')
axes[2].legend()

plt.tight_layout()
plt.savefig('dataset_overview.png', bbox_inches='tight')
plt.show()
print("✅ Dataset visualisation saved.")


## 3. Patient & Session Simulation
Generate synthetic session history for 50 patients across 30 sessions each.
This data trains and validates the recommendation models.

In [ ]:
random.seed(42)
np.random.seed(42)

DIFFICULTY_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
DIFFICULTY_INV = {0: 'Low', 1: 'Medium', 2: 'High'}

# ── Score bands ───────────────────────────────────────────────
def get_difficulty_tier(cumulative_score):
    if cumulative_score <= 30:
        return 'Low'
    elif cumulative_score <= 70:
        return 'Medium'
    else:
        return 'High'

# ── Simulate one session ──────────────────────────────────────
def simulate_session(patient_profile, activity_row):
    """
    Simulate patient performance on a given activity.
    Patient profile controls baseline skill level.
    """
    base_skill   = patient_profile['skill_level']   # 0–1
    fatigue      = patient_profile['fatigue']        # 0–1 (higher = more fatigued)
    diff_penalty = {'Low': 0, 'Medium': 0.15, 'High': 0.30}[activity_row['difficulty_level']]

    # Raw metrics
    accuracy   = np.clip(np.random.normal(base_skill - diff_penalty - fatigue*0.1, 0.12), 0, 1) * 100
    speed_ratio= np.clip(np.random.normal(0.85 - diff_penalty*0.5, 0.15), 0.3, 1.5)
    speed_score= min(100, speed_ratio * 100)
    streak_len = max(1, int(accuracy / 100 * np.random.randint(5, 15)))
    total_q    = random.randint(8, 15)
    consistency= min(100, streak_len / total_q * 100)
    hints_used = max(0, int(np.random.poisson(2 * (1 - base_skill + diff_penalty))))
    wrong_att  = max(0, int(np.random.poisson(3 * (1 - accuracy/100))))
    retries    = max(0, int(accuracy < 40) * random.randint(0, 4))

    # Penalise accuracy for hints
    adj_accuracy = max(0, accuracy - hints_used * 5)

    # Composite score
    composite = (adj_accuracy * 0.5 + speed_score * 0.3 + consistency * 0.2)
    composite = round(composite, 2)

    return {
        'accuracy':      round(adj_accuracy, 2),
        'speed_score':   round(speed_score, 2),
        'consistency':   round(consistency, 2),
        'hints_used':    hints_used,
        'wrong_attempts':wrong_att,
        'retries':       retries,
        'actual_time':   round(activity_row['expected_time_sec'] / max(0.3, speed_ratio), 1),
        'composite_score': composite
    }

print("✅ Session simulation functions defined.")


In [ ]:
# ── Generate full patient history ─────────────────────────────
records = []

for pid in range(1, 51):   # 50 patients
    skill_level  = np.clip(np.random.normal(0.65, 0.18), 0.2, 0.95)
    fatigue_base = np.clip(np.random.normal(0.15, 0.08), 0.0, 0.40)
    cum_score    = 0.0
    score_history= []

    for session_num in range(1, 31):   # 30 sessions each
        tier        = get_difficulty_tier(cum_score)
        candidates  = df[df['difficulty_level'] == tier]
        activity    = candidates.sample(1).iloc[0]
        fatigue     = fatigue_base + (session_num / 60)   # fatigue grows slowly

        perf = simulate_session(
            {'skill_level': skill_level, 'fatigue': fatigue},
            activity
        )
        cum_score += perf['composite_score'] * 0.1   # damped accumulation

        # Improvement from previous session
        improvement = 0.0
        if score_history:
            prev = score_history[-1]
            improvement = ((perf['composite_score'] - prev) / max(1, prev)) * 100

        # Weak category detection (mock)
        weak_cat = random.choice(df['category'].unique()) if random.random() < 0.3 else ''

        records.append({
            'patient_id':         f"P-{pid:04d}",
            'session_num':        session_num,
            'activity_id':        activity['activity_id'],
            'activity_name':      activity['activity_name'],
            'category':           activity['category'],
            'difficulty_level':   activity['difficulty_level'],
            'cumulative_score':   round(cum_score, 2),
            **perf,
            'improvement_pct':    round(improvement, 2),
            'weak_category':      weak_cat,
            'skill_level_true':   round(skill_level, 3)
        })
        score_history.append(perf['composite_score'])

history_df = pd.DataFrame(records)
print(f"✅ Generated {len(history_df):,} session records")
print(f"   Patients : {history_df['patient_id'].nunique()}")
print(f"   Sessions : {history_df['session_num'].max()} per patient")
history_df.head(3)


## 4. Scoring Engine
The composite score formula and all level-progression rules.

In [ ]:
# ── Core scoring formula ──────────────────────────────────────
def compute_composite_score(accuracy, speed_score, consistency,
                             hints_used=0, acc_w=0.5, spd_w=0.3, con_w=0.2):
    """
    Score = (Accuracy × acc_w) + (Speed × spd_w) + (Consistency × con_w)
    Hint penalty: 5 points per hint used.
    All inputs are 0–100 scale.
    """
    adj_acc = max(0, min(100, accuracy - hints_used * 5))
    score   = (adj_acc * acc_w) + (speed_score * spd_w) + (consistency * con_w)
    return round(score, 2)

# ── Validate against known values ────────────────────────────
examples = [
    (80, 70, 90, 0),    # good session
    (55, 50, 60, 3),    # average with hints
    (30, 40, 20, 8),    # poor session
]
print("=== Formula validation ===")
print(f"{'Acc':>5} {'Spd':>5} {'Con':>5} {'Hints':>6}  →  {'Score':>6}")
print("-" * 40)
for acc, spd, con, h in examples:
    s = compute_composite_score(acc, spd, con, h)
    print(f"{acc:>5} {spd:>5} {con:>5} {h:>6}  →  {s:>6}")


In [ ]:
# ── Level progression engine ──────────────────────────────────
class PatientState:
    def __init__(self, patient_id):
        self.patient_id       = patient_id
        self.cumulative_score = 0.0
        self.score_history    = []
        self.difficulty       = 'Low'
        self.consecutive_high = 0
        self.consecutive_low  = 0
        self.retry_count      = 0
        self.caregiver_alert  = False

    def update(self, composite_score, retry_count=0):
        self.cumulative_score += composite_score * 0.1
        self.score_history.append(composite_score)
        self.retry_count = retry_count

        # Promotion: 3 consecutive sessions ≥ 80
        if composite_score >= 80:
            self.consecutive_high += 1
            self.consecutive_low   = 0
        else:
            self.consecutive_high = 0

        if self.consecutive_high >= 3:
            self.difficulty = self._promote(self.difficulty)
            self.consecutive_high = 0

        # Demotion: 2 consecutive sessions < 40
        if composite_score < 40:
            self.consecutive_low += 1
            self.consecutive_high = 0
        else:
            self.consecutive_low = 0

        if self.consecutive_low >= 2:
            self.difficulty = self._demote(self.difficulty)
            self.consecutive_low = 0

        # Fallback: retries ≥ 3 → demote for next pick
        if retry_count >= 3:
            self.difficulty = self._demote(self.difficulty)
            self.retry_count = 0

        # Caregiver alert: ≥20% weekly decline
        self._check_alert()
        return self

    def _promote(self, d):
        return {'Low': 'Medium', 'Medium': 'High', 'High': 'High'}[d]

    def _demote(self, d):
        return {'High': 'Medium', 'Medium': 'Low', 'Low': 'Low'}[d]

    def _check_alert(self):
        h = self.score_history
        if len(h) < 7:
            self.caregiver_alert = False
            return
        week_avg = mean(h[-7:])
        prev_avg = mean(h[-14:-7]) if len(h) >= 14 else mean(h[:-7]) if len(h) > 7 else week_avg
        if prev_avg > 0:
            decline = (prev_avg - week_avg) / prev_avg * 100
            self.caregiver_alert = decline >= 20
        return self.caregiver_alert

# ── Quick demo ────────────────────────────────────────────────
state = PatientState("P-DEMO")
scores = [72, 78, 82, 85, 88, 30, 28, 75, 80, 83, 85]
print("Session  Score   Difficulty   Alert")
print("-" * 42)
for i, s in enumerate(scores, 1):
    state.update(s)
    alert = "⚠️  YES" if state.caregiver_alert else "—"
    print(f"  {i:>3}    {s:>5}   {state.difficulty:<10}   {alert}")


In [ ]:
# ── Visualise score progression & difficulty band ─────────────
sample_patient = history_df[history_df['patient_id'] == 'P-0001'].copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(sample_patient['session_num'], sample_patient['composite_score'],
         marker='o', ms=5, linewidth=1.5, color='#1976D2', label='Composite score')
ax1.axhline(80, color='green',  linestyle='--', alpha=0.6, label='High threshold (80)')
ax1.axhline(40, color='orange', linestyle='--', alpha=0.6, label='Low threshold (40)')
ax1.set_ylabel('Score (0–100)')
ax1.set_title('Patient P-0001 — Session Scores & Difficulty')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 105)

diff_colors = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
bars = ax2.bar(
    sample_patient['session_num'],
    [1]*len(sample_patient),
    color=[diff_colors[d] for d in sample_patient['difficulty_level']],
    edgecolor='none'
)
ax2.set_yticks([])
ax2.set_ylabel('Difficulty')
ax2.set_xlabel('Session number')
patches = [mpatches.Patch(color=c, label=l) for l, c in diff_colors.items()]
ax2.legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig('patient_progression.png', bbox_inches='tight')
plt.show()


## 5. Feature Engineering
Six engineered features computed from raw session history feed the ML models.

In [ ]:
def engineer_features(patient_history, window=5):
    """
    Compute the 15-dimensional feature vector for the ML models.
    patient_history: list of session dicts, sorted oldest → newest.
    """
    if not patient_history:
        return {}

    recent = patient_history[-window:]

    # ── Raw recent metrics ────────────────────────────────────
    accs   = [s['accuracy']      for s in recent]
    spds   = [s['speed_score']   for s in recent]
    cons   = [s['consistency']   for s in recent]
    hints  = [s['hints_used']    for s in recent]
    wrongs = [s['wrong_attempts']for s in recent]
    rets   = [s['retries']       for s in recent]

    # ── Engineered ────────────────────────────────────────────
    rolling_avg_acc = mean(accs)

    # Speed trend: slope via least squares
    if len(spds) >= 2:
        x = np.arange(len(spds), dtype=float)
        speed_trend = float(np.polyfit(x, spds, 1)[0])
    else:
        speed_trend = 0.0

    # Preferred category (most frequent in history)
    all_cats = [s.get('category', '') for s in patient_history]
    cat_counts = defaultdict(int)
    for c in all_cats:
        cat_counts[c] += 1
    preferred_cat = max(cat_counts, key=cat_counts.get, default='')

    # Category weakness score (global avg − category avg)
    cat_acc = defaultdict(list)
    for s in patient_history:
        cat_acc[s.get('category','')].append(s['accuracy'])
    global_avg = mean([s['accuracy'] for s in patient_history]) if patient_history else 50
    weakness_score = 0.0
    weak_cat_name  = ''
    for cat, vals in cat_acc.items():
        diff = global_avg - mean(vals)
        if diff > weakness_score:
            weakness_score = diff
            weak_cat_name  = cat

    # Level tolerance: std dev of scores at current difficulty
    cur_diff = patient_history[-1].get('difficulty_level', 'Low')
    diff_scores = [s['composite_score'] for s in patient_history
                   if s.get('difficulty_level') == cur_diff]
    level_tolerance = stdev(diff_scores) if len(diff_scores) >= 2 else 0.0

    # Session fatigue: final 20% vs first 80%
    all_accs = [s['accuracy'] for s in patient_history]
    split    = max(1, int(len(all_accs) * 0.8))
    fatigue_score = (mean(all_accs[split:]) / mean(all_accs[:split])
                     if all_accs[:split] else 1.0)

    # Consecutive high / low sessions
    scores_rev = [s['composite_score'] for s in reversed(patient_history)]
    consec_high = 0
    for sc in scores_rev:
        if sc >= 80: consec_high += 1
        else: break
    consec_low = 0
    for sc in scores_rev:
        if sc < 40: consec_low += 1
        else: break

    return {
        'session_accuracy':       round(accs[-1], 2),
        'session_speed':          round(spds[-1], 2),
        'session_consistency':    round(cons[-1], 2),
        'hint_usage_rate':        round(mean(hints) / 10, 3),
        'retry_count':            int(rets[-1]),
        'wrong_attempts':         int(wrongs[-1]),
        'rolling_avg_accuracy':   round(rolling_avg_acc, 2),
        'speed_trend':            round(speed_trend, 4),
        'preferred_category':     preferred_cat,
        'weakness_score':         round(weakness_score, 2),
        'weak_category':          weak_cat_name,
        'level_tolerance':        round(level_tolerance, 2),
        'fatigue_score':          round(fatigue_score, 3),
        'consecutive_high':       consec_high,
        'consecutive_low':        consec_low,
        'current_difficulty':     cur_diff
    }

# ── Demo on one patient ───────────────────────────────────────
p1 = history_df[history_df['patient_id'] == 'P-0001'].to_dict('records')
feats = engineer_features(p1)
print("=== Feature vector for P-0001 (all sessions) ===\n")
for k, v in feats.items():
    print(f"  {k:<30} : {v}")


In [ ]:
# ── Build feature matrix for all patients ─────────────────────
feature_rows = []

for pid, grp in history_df.groupby('patient_id'):
    sessions = grp.sort_values('session_num').to_dict('records')
    for i in range(5, len(sessions)):           # need ≥5 sessions of context
        past_sessions = sessions[:i]
        target_session = sessions[i]
        feat = engineer_features(past_sessions)
        feat['patient_id']     = pid
        feat['target_difficulty'] = target_session['difficulty_level']
        feat['target_activity']   = target_session['activity_id']
        feature_rows.append(feat)

feat_df = pd.DataFrame(feature_rows)
print(f"✅ Feature matrix: {feat_df.shape[0]:,} rows × {feat_df.shape[1]} columns")
feat_df.head(3)


## 6. Phase 1 — Rule-Based Recommender
Deterministic recommender used during cold start (fewer than 10 sessions).

In [ ]:
def rule_based_recommend(patient_id, history_df, activity_df):
    """
    Phase 1 recommender.
    Uses score bands and category weakness to select the next activity.
    Always starts with Low for new patients.
    """
    patient_sessions = history_df[history_df['patient_id'] == patient_id].sort_values('session_num')

    # Cold start: no history
    if len(patient_sessions) == 0:
        candidates = activity_df[activity_df['difficulty_level'] == 'Low']
        chosen = candidates.sample(1).iloc[0]
        return {
            'recommended_activity': chosen['activity_name'],
            'activity_id':          chosen['activity_id'],
            'difficulty':           'Low',
            'confidence':           1.0,
            'reason':               'Cold start — initial Low difficulty assigned.',
            'phase':                'rule-based'
        }

    cum_score = patient_sessions['cumulative_score'].iloc[-1]
    tier      = get_difficulty_tier(cum_score)

    # Weak category override
    cat_acc = patient_sessions.groupby('category')['accuracy'].mean()
    global_acc = patient_sessions['accuracy'].mean()
    weak_cats = cat_acc[cat_acc < global_acc - 10]
    weak_cat  = weak_cats.idxmin() if len(weak_cats) > 0 else None

    # Retry fallback
    last_retries = patient_sessions['retries'].iloc[-1]
    if last_retries >= 3:
        tier = {'High': 'Medium', 'Medium': 'Low', 'Low': 'Low'}[tier]

    candidates = activity_df[activity_df['difficulty_level'] == tier]
    if weak_cat:
        targeted = candidates[candidates['category'] == weak_cat]
        if len(targeted) > 0:
            candidates = targeted

    # Avoid repeating last 3 activities
    recent_acts = patient_sessions['activity_id'].tail(3).tolist()
    fresh = candidates[~candidates['activity_id'].isin(recent_acts)]
    pool  = fresh if len(fresh) > 0 else candidates

    chosen = pool.sample(1).iloc[0]
    reason_parts = [f"Cumulative score {cum_score:.1f} → {tier} tier"]
    if weak_cat:
        reason_parts.append(f"Targeting weak category: {weak_cat}")
    if last_retries >= 3:
        reason_parts.append(f"Difficulty reduced due to {last_retries} retries")

    return {
        'recommended_activity': chosen['activity_name'],
        'activity_id':          chosen['activity_id'],
        'difficulty':           tier,
        'confidence':           0.75,
        'reason':               '; '.join(reason_parts),
        'phase':                'rule-based'
    }

# ── Demo ──────────────────────────────────────────────────────
for pid in ['P-0001', 'P-0015', 'P-0030']:
    rec = rule_based_recommend(pid, history_df, df)
    print(f"\nPatient {pid}")
    for k, v in rec.items():
        print(f"  {k:<26}: {v}")


## 7. Phase 2 — ML Models
Random Forest for difficulty classification and XGBoost for activity ranking.
Activated once ≥10 sessions of history exist per patient.

In [ ]:
# ── Prepare ML feature matrix ─────────────────────────────────
NUMERIC_FEATURES = [
    'session_accuracy', 'session_speed', 'session_consistency',
    'hint_usage_rate', 'retry_count', 'wrong_attempts',
    'rolling_avg_accuracy', 'speed_trend', 'weakness_score',
    'level_tolerance', 'fatigue_score', 'consecutive_high', 'consecutive_low'
]

le_diff = LabelEncoder()
le_cat  = LabelEncoder()
le_diff.fit(['Low', 'Medium', 'High'])

# Fill and encode
feat_df_ml = feat_df.dropna(subset=NUMERIC_FEATURES + ['target_difficulty']).copy()
feat_df_ml['target_difficulty_enc'] = le_diff.transform(feat_df_ml['target_difficulty'])
feat_df_ml['current_difficulty_enc']= le_diff.transform(
    feat_df_ml['current_difficulty'].map(lambda x: x if x in ['Low','Medium','High'] else 'Low'))

X = feat_df_ml[NUMERIC_FEATURES + ['current_difficulty_enc']].values
y = feat_df_ml['target_difficulty_enc'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {le_diff.classes_}")


In [ ]:
# ── Random Forest — Difficulty Classifier ─────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("=== Random Forest — Difficulty Classifier ===\n")
print(classification_report(y_test, y_pred, target_names=le_diff.classes_))

# Cross-validation
cv_scores = cross_val_score(rf, X_scaled, y, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


In [ ]:
# ── Feature importance ────────────────────────────────────────
feature_names = NUMERIC_FEATURES + ['current_difficulty_enc']
importances   = rf.feature_importances_
idx           = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(importances)), importances[idx], color='steelblue', edgecolor='none')
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_names[i] for i in idx], rotation=45, ha='right', fontsize=9)
ax.set_title('Random Forest — Feature Importances')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_diff.classes_,
            yticklabels=le_diff.classes_, ax=ax)
ax.set_title('Confusion Matrix — Difficulty Prediction')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── XGBoost — Activity Ranker (if available) ──────────────────
if XGB_AVAILABLE:
    from sklearn.preprocessing import LabelEncoder as LE2

    le_act = LE2()
    feat_df_ml['target_activity_enc'] = le_act.fit_transform(feat_df_ml['target_activity'])
    y_act = feat_df_ml['target_activity_enc'].values

    Xa_train, Xa_test, ya_train, ya_test = train_test_split(
        X_scaled, y_act, test_size=0.2, random_state=42)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
    xgb_clf.fit(Xa_train, ya_train,
                eval_set=[(Xa_test, ya_test)],
                verbose=False)

    xgb_acc = xgb_clf.score(Xa_test, ya_test)
    print(f"✅ XGBoost activity classifier accuracy: {xgb_acc:.3f}")
else:
    print("⚠️  XGBoost not available. Install with: pip install xgboost")
    xgb_clf = None


## 8. Adaptive Recommendation Pipeline
Combines Phase 1 (rule-based) and Phase 2 (ML) into one unified API function.

In [ ]:
def ml_recommend(patient_id, history_df, activity_df,
                  rf_model, scaler, le_diff, feature_names):
    """
    Full recommendation pipeline.
    Phases:
      - <10 sessions → rule-based
      - ≥10 sessions → Random Forest difficulty + activity selection
    Returns API-ready dict.
    """
    patient_sessions = history_df[history_df['patient_id'] == patient_id].sort_values('session_num')
    n_sessions = len(patient_sessions)

    # ── Phase 1 cold start ────────────────────────────────────
    if n_sessions < 10:
        result = rule_based_recommend(patient_id, history_df, activity_df)
        result['caregiver_alert'] = False
        result['session_length_minutes'] = 8
        return result

    # ── Phase 2 ML ────────────────────────────────────────────
    sessions_list = patient_sessions.to_dict('records')
    feats = engineer_features(sessions_list)

    numeric_vals = [
        feats.get('session_accuracy', 0),
        feats.get('session_speed', 0),
        feats.get('session_consistency', 0),
        feats.get('hint_usage_rate', 0),
        feats.get('retry_count', 0),
        feats.get('wrong_attempts', 0),
        feats.get('rolling_avg_accuracy', 0),
        feats.get('speed_trend', 0),
        feats.get('weakness_score', 0),
        feats.get('level_tolerance', 0),
        feats.get('fatigue_score', 1),
        feats.get('consecutive_high', 0),
        feats.get('consecutive_low', 0),
    ]
    cur_diff_enc = le_diff.transform([feats.get('current_difficulty', 'Low')])[0]
    X_input = scaler.transform([numeric_vals + [cur_diff_enc]])

    diff_enc   = rf_model.predict(X_input)[0]
    proba      = rf_model.predict_proba(X_input)[0]
    confidence = float(proba.max())
    tier       = le_diff.inverse_transform([diff_enc])[0]

    # ── Select activity ───────────────────────────────────────
    candidates = activity_df[activity_df['difficulty_level'] == tier]
    weak_cat   = feats.get('weak_category', '')
    if weak_cat:
        targeted = candidates[candidates['category'] == weak_cat]
        if len(targeted) > 0:
            candidates = targeted

    recent_acts = patient_sessions['activity_id'].tail(3).tolist()
    fresh = candidates[~candidates['activity_id'].isin(recent_acts)]
    pool  = fresh if len(fresh) > 0 else candidates
    chosen = pool.sample(1).iloc[0]

    # ── Reason generation ─────────────────────────────────────
    reasons = []
    ra = feats.get('rolling_avg_accuracy', 0)
    if ra >= 75:
        reasons.append(f"Strong rolling accuracy ({ra:.0f}%)")
    elif ra < 50:
        reasons.append(f"Low rolling accuracy ({ra:.0f}%) — difficulty held or reduced")
    st = feats.get('speed_trend', 0)
    if st > 1:
        reasons.append("Improving speed trend")
    elif st < -1:
        reasons.append("Declining speed — extra time activity preferred")
    if weak_cat:
        reasons.append(f"Targeting weak area: {weak_cat}")
    fs = feats.get('fatigue_score', 1)
    if fs < 0.75:
        reasons.append("Fatigue detected — shorter session recommended")
    if not reasons:
        reasons.append("Stable progression — difficulty maintained")

    # ── Caregiver alert ───────────────────────────────────────
    state = PatientState(patient_id)
    for sc in patient_sessions['composite_score'].tolist():
        state.update(sc)
    alert = state.caregiver_alert

    session_len = {'Low': 8, 'Medium': 12, 'High': 18}[tier]
    if fs < 0.75:
        session_len = max(5, session_len - 4)

    return {
        'patient_id':              patient_id,
        'recommended_activity':    chosen['activity_name'],
        'activity_id':             chosen['activity_id'],
        'difficulty':              tier,
        'confidence':              round(confidence, 3),
        'reason':                  '; '.join(reasons),
        'phase':                   'ml',
        'caregiver_alert':         alert,
        'session_length_minutes':  session_len,
        'next_review_after_sessions': 3,
        'weak_category':           weak_cat or None
    }

print("✅ ml_recommend() defined.")


In [ ]:
# ── Run recommendations for a batch of patients ───────────────
results = []
sample_patients = history_df['patient_id'].unique()[:10]

for pid in sample_patients:
    rec = ml_recommend(pid, history_df, df, rf, scaler, le_diff, NUMERIC_FEATURES)
    results.append(rec)

rec_df = pd.DataFrame(results)
print("=== Batch recommendations ===\n")
print(rec_df[['patient_id','recommended_activity','difficulty','confidence',
              'caregiver_alert','session_length_minutes']].to_string(index=False))


In [ ]:
# ── Pretty-print one full API response ───────────────────────
sample_rec = ml_recommend('P-0001', history_df, df, rf, scaler, le_diff, NUMERIC_FEATURES)
print(json.dumps(sample_rec, indent=2))


## 9. Caregiver Alert Dashboard
Flag patients showing significant weekly performance decline.

In [ ]:
def generate_alert_report(history_df, threshold_pct=20):
    """
    Check all patients for significant performance decline.
    Returns a DataFrame with alert status and statistics.
    """
    report = []
    for pid, grp in history_df.groupby('patient_id'):
        scores = grp.sort_values('session_num')['composite_score'].tolist()
        if len(scores) < 7:
            continue
        week_avg = mean(scores[-7:])
        prev_avg = mean(scores[-14:-7]) if len(scores) >= 14 else mean(scores[:-7])
        if prev_avg > 0:
            decline = (prev_avg - week_avg) / prev_avg * 100
        else:
            decline = 0.0
        report.append({
            'patient_id':   pid,
            'week_avg':     round(week_avg, 1),
            'prev_avg':     round(prev_avg, 1),
            'decline_pct':  round(decline, 1),
            'alert':        decline >= threshold_pct,
            'total_sessions': len(scores)
        })
    return pd.DataFrame(report)

alert_report = generate_alert_report(history_df)
alerted = alert_report[alert_report['alert'] == True]

print(f"Total patients monitored : {len(alert_report)}")
print(f"Caregiver alerts raised  : {len(alerted)} ({len(alerted)/len(alert_report)*100:.1f}%)\n")
print(alerted[['patient_id','week_avg','prev_avg','decline_pct']].head(10).to_string(index=False))


In [ ]:
# ── Visualise decline distribution ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(alert_report['decline_pct'], bins=20, color='steelblue', edgecolor='white')
ax1.axvline(20, color='red', linestyle='--', label='Alert threshold (20%)')
ax1.set_xlabel('Decline %')
ax1.set_ylabel('Count')
ax1.set_title('Performance Decline Distribution')
ax1.legend()

colors = ['#F44336' if a else '#4CAF50' for a in alert_report['alert']]
ax2.scatter(alert_report['prev_avg'], alert_report['week_avg'],
            c=colors, alpha=0.7, s=50, edgecolors='none')
ax2.plot([0, 100], [0, 100], 'k--', alpha=0.3)
ax2.set_xlabel('Previous week avg score')
ax2.set_ylabel('Current week avg score')
ax2.set_title('Week-on-Week Performance')
red_patch   = mpatches.Patch(color='#F44336', label='Alert')
green_patch = mpatches.Patch(color='#4CAF50', label='Stable')
ax2.legend(handles=[red_patch, green_patch])

plt.tight_layout()
plt.savefig('caregiver_alerts.png', bbox_inches='tight')
plt.show()


## 10. Export Results

In [ ]:
# ── Save all outputs ──────────────────────────────────────────
history_df.to_csv('patient_session_history.csv', index=False)
alert_report.to_csv('caregiver_alert_report.csv', index=False)
rec_df.to_csv('batch_recommendations.csv', index=False)

print("✅ Files saved:")
print("   alzheimer_activities.csv       — activity dataset")
print("   patient_session_history.csv    — simulated 50-patient × 30-session history")
print("   caregiver_alert_report.csv     — alert status for all patients")
print("   batch_recommendations.csv      — sample recommendations")
print("   dataset_overview.png")
print("   patient_progression.png")
print("   feature_importance.png")
print("   confusion_matrix.png")
print("   caregiver_alerts.png")
